In [1]:
import os
import re
import time
import json
import math
import unicodedata
from difflib import SequenceMatcher

import requests
import pandas as pd
from tqdm.auto import tqdm

from pathlib import Path

/home/hbli/songformer/env/miniforge3/envs/songformer/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### HX

In [ ]:
METADATA_PATH = "/mnt/ssd/hbli/datasets/harmonixset/metadata.csv"
OUTPUT_CSV = "/mnt/ssd/hbli/datasets/harmonixset/harmonixset_lrclib_results.csv"
OUTPUT_JSONL = "/mnt/ssd/hbli/datasets/harmonixset/harmonixset_lrclib_results.jsonl"

CHECKPOINT_EVERY = 50            # save once after per CHECKPOINT_EVERY
SLEEP_BETWEEN_REQUESTS = 0.25    # limit speed
TIMEOUT = 20
MAX_RETRIES = 3

MIN_ACCEPT_SCORE = 0.72

LRCLIB_GET_URL = "https://lrclib.net/api/get"
LRCLIB_SEARCH_URL = "https://lrclib.net/api/search"

In [ ]:
# metadata, use cols
df = pd.read_csv(METADATA_PATH)

required_cols = ["File", "Title", "Artist", "Release", "Duration"]
work_df = df[required_cols].copy()

for col in ["File", "Title", "Artist", "Release"]:
    work_df[col] = work_df[col].fillna("").astype(str).str.strip()

work_df["Duration"] = pd.to_numeric(work_df["Duration"], errors="coerce")
work_df = work_df.dropna(subset=["Duration"]).reset_index(drop=True)
print("total count", len(work_df))

In [ ]:
def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))

def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text).lower().strip()
    text = strip_accents(text)

    # delete such as (Remastered), [Live], (feat. xxx)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\[[^\]]*\]", " ", text)

    noise_patterns = [
        r"\bfeat\.?.*$",
        r"\bft\.?.*$",
        r"\bfeaturing\b.*$",
        r"\bremaster(ed)?\b",
        r"\blive\b",
        r"\bversion\b",
        r"\bedit\b",
        r"\bmix\b",
        r"\bmono\b",
        r"\bstereo\b",
        r"\bexplicit\b",
        r"\bdeluxe\b",
    ]
    for p in noise_patterns:
        text = re.sub(p, " ", text)

    # delete punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def similarity(a: str, b: str) -> float:
    a = normalize_text(a)
    b = normalize_text(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

In [ ]:
def sec_to_api_duration(x):
    if pd.isna(x):
        return None
    return int(round(float(x)))

def normalize_duration_to_sec(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None

    x = float(x)

    # if > 10000，must be ms
    if x > 10000:
        return x / 1000.0
    return x

In [ ]:
session = requests.Session()

def request_json(url, params, max_retries=MAX_RETRIES, timeout=TIMEOUT):
    last_error = None

    for attempt in range(max_retries):
        try:
            resp = session.get(url, params=params, timeout=timeout)

            if resp.status_code == 404:
                return None

            if resp.status_code == 429:
                # wait when 429
                wait = 2 ** attempt
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()

        except requests.RequestException as e:
            last_error = str(e)
            wait = 2 ** attempt
            time.sleep(wait)

    print(f"[WARN] fail request: {url}, params={params}, err={last_error}")
    return None

In [ ]:
# accurate fetch
def lrclib_get(title, artist, release=None, duration_sec=None):
    """
    title + artist + duration
    """
    params = {
        "track_name": title,
        "artist_name": artist,
    }

    if release:
        params["album_name"] = release

    if duration_sec is not None and not pd.isna(duration_sec):
        params["duration"] = sec_to_api_duration(duration_sec)

    data = request_json(LRCLIB_GET_URL, params)
    time.sleep(SLEEP_BETWEEN_REQUESTS)
    return data

In [ ]:
# fuzzy match
def lrclib_search(title, artist, release=None, duration_sec=None):
    """
    if fail in get, use search api
    """
    params = {
        "track_name": title,
        "artist_name": artist,
        "q": f"{title} {artist}".strip()
    }

    if release:
        params["album_name"] = release

    if duration_sec is not None and not pd.isna(duration_sec):
        params["duration"] = sec_to_api_duration(duration_sec)

    data = request_json(LRCLIB_SEARCH_URL, params)
    time.sleep(SLEEP_BETWEEN_REQUESTS)

    if data is None:
        return []
    if isinstance(data, list):
        return data
    return []

In [ ]:
def get_field(d, *keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None

def score_candidate(row, cand):
    """
    score for result, depends on title, artist, album, duration
    """
    cand_title = get_field(cand, "trackName", "track_name", "name", "title")
    cand_artist = get_field(cand, "artistName", "artist_name", "artist")
    cand_album = get_field(cand, "albumName", "album_name", "album")
    cand_duration = get_field(cand, "duration")

    title_score = similarity(row["Title"], cand_title)
    artist_score = similarity(row["Artist"], cand_artist)
    album_score = similarity(row["Release"], cand_album) if row["Release"] else 0.0

    row_dur = float(row["Duration"])
    cand_dur = normalize_duration_to_sec(cand_duration)
    if cand_dur is None:
        duration_score = 0.0
        duration_diff = None
    else:
        duration_diff = abs(row_dur - cand_dur)
        # >15, 0
        duration_score = max(0.0, 1.0 - min(duration_diff, 15.0) / 15.0)

    final_score = (
        0.50 * title_score +
        0.30 * artist_score +
        0.15 * album_score +
        0.05 * duration_score
    )

    return {
        "final_score": final_score,
        "title_score": title_score,
        "artist_score": artist_score,
        "album_score": album_score,
        "duration_score": duration_score,
        "duration_diff_sec": duration_diff,
    }

In [ ]:
def build_result_row(row, matched, method, score_info=None, status="matched", error_msg=None):
    if matched is None:
        return {
            "File": row["File"],
            "Title": row["Title"],
            "Artist": row["Artist"],
            "Release": row["Release"],
            "Duration": row["Duration"],
            "status": status,
            "match_method": method,
            "match_score": None if score_info is None else score_info.get("final_score"),
            "title_score": None if score_info is None else score_info.get("title_score"),
            "artist_score": None if score_info is None else score_info.get("artist_score"),
            "album_score": None if score_info is None else score_info.get("album_score"),
            "duration_score": None if score_info is None else score_info.get("duration_score"),
            "duration_diff_sec": None if score_info is None else score_info.get("duration_diff_sec"),
            "lrclib_id": None,
            "matched_track": None,
            "matched_artist": None,
            "matched_album": None,
            "matched_duration": None,
            "instrumental": None,
            "plain_lyrics": None,
            "synced_lyrics": None,
            "has_plain": False,
            "has_synced": False,
            "error": error_msg,
        }

    lrclib_id = get_field(matched, "id")
    cand_title = get_field(matched, "trackName", "track_name", "name", "title")
    cand_artist = get_field(matched, "artistName", "artist_name", "artist")
    cand_album = get_field(matched, "albumName", "album_name", "album")
    cand_duration = get_field(matched, "duration")
    instrumental = get_field(matched, "instrumental")
    plain_lyrics = get_field(matched, "plainLyrics", "plain_lyrics")
    synced_lyrics = get_field(matched, "syncedLyrics", "synced_lyrics")

    return {
        "File": row["File"],
        "Title": row["Title"],
        "Artist": row["Artist"],
        "Release": row["Release"],
        "Duration": row["Duration"],
        "status": status,
        "match_method": method,
        "match_score": None if score_info is None else score_info.get("final_score"),
        "title_score": None if score_info is None else score_info.get("title_score"),
        "artist_score": None if score_info is None else score_info.get("artist_score"),
        "album_score": None if score_info is None else score_info.get("album_score"),
        "duration_score": None if score_info is None else score_info.get("duration_score"),
        "duration_diff_sec": None if score_info is None else score_info.get("duration_diff_sec"),
        "lrclib_id": lrclib_id,
        "matched_track": cand_title,
        "matched_artist": cand_artist,
        "matched_album": cand_album,
        "matched_duration": normalize_duration_to_sec(cand_duration),
        "instrumental": instrumental,
        "plain_lyrics": plain_lyrics,
        "synced_lyrics": synced_lyrics,
        "has_plain": bool(plain_lyrics),
        "has_synced": bool(synced_lyrics),
        "error": error_msg,
    }


def resolve_one_song(row):
    # ---------- Step 1: exact get ----------
    exact = lrclib_get(
        title=row["Title"],
        artist=row["Artist"],
        release=row["Release"],
        duration_sec=row["Duration"]
    )

    if exact:
        exact_score = score_candidate(row, exact)
        if exact_score["final_score"] >= MIN_ACCEPT_SCORE:
            return build_result_row(
                row=row,
                matched=exact,
                method="get_exact",
                score_info=exact_score,
                status="matched"
            )

    # ---------- Step 2: search fallback ----------
    candidates = lrclib_search(
        title=row["Title"],
        artist=row["Artist"],
        release=row["Release"],
        duration_sec=row["Duration"]
    )

    if not candidates:
        return build_result_row(
            row=row,
            matched=None,
            method="search_fallback",
            score_info=None,
            status="not_found"
        )

    scored = []
    for cand in candidates:
        s = score_candidate(row, cand)
        scored.append((cand, s))

    scored.sort(key=lambda x: x[1]["final_score"], reverse=True)
    best_cand, best_score = scored[0]

    status = "matched" if best_score["final_score"] >= MIN_ACCEPT_SCORE else "low_confidence"

    return build_result_row(
        row=row,
        matched=best_cand,
        method="search_fallback",
        score_info=best_score,
        status=status
    )

In [ ]:
if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    done_files = set(existing["File"].astype(str).tolist())
    print(f"already have {len(existing)}, skip")
else:
    existing = pd.DataFrame()
    done_files = set()

todo_df = work_df[~work_df["File"].astype(str).isin(done_files)].reset_index(drop=True)
print("wait for processing：", len(todo_df))
display(todo_df.head())

In [ ]:
results = []

for idx, row in tqdm(todo_df.iterrows(), total=len(todo_df)):
    try:
        out = resolve_one_song(row)
    except Exception as e:
        out = build_result_row(
            row=row,
            matched=None,
            method="internal_error",
            score_info=None,
            status="error",
            error_msg=str(e)
        )

    results.append(out)

    if (idx + 1) % CHECKPOINT_EVERY == 0 or (idx + 1) == len(todo_df):
        new_df = pd.DataFrame(results)

        if len(existing) > 0:
            save_df = pd.concat([existing, new_df], ignore_index=True)
        else:
            save_df = new_df.copy()

        save_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

        with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
            for _, r in save_df.iterrows():
                f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

        print(f"[checkpoint] have saved to {OUTPUT_CSV} / {OUTPUT_JSONL}")

In [ ]:
# final result
final_df = pd.read_csv(OUTPUT_CSV)

In [ ]:
final_df['status'].value_counts()

In [ ]:
final_df[final_df['status'] == 'low_confidence']

In [ ]:
final_df[final_df['status'] == 'not_found']

### Hook

In [ ]:
def score_candidate(row, cand):
    cand_title = get_field(cand, "trackName", "track_name", "name", "title")
    cand_artist = get_field(cand, "artistName", "artist_name", "artist")

    title_score = similarity(row["Title"], cand_title)
    artist_score = similarity(row["Artist"], cand_artist)

    final_score = 0.5 * title_score + 0.5 * artist_score

    return {
        "final_score": final_score,
        "title_score": title_score,
        "artist_score": artist_score,
        "album_score": None,
        "duration_score": None,
        "duration_diff_sec": None,
    }

In [ ]:
def build_result_row(row, matched, method, score_info=None, status="matched", error_msg=None):
    base = {
        "File": row["File"],
        "AudioPath": row.get("AudioPath", None),
        "Title": row["Title"],
        "Artist": row["Artist"],
        "Release": row.get("Release", ""),
        "Duration": row.get("Duration", None),
        "status": status,
        "match_method": method,
        "match_score": None if score_info is None else score_info.get("final_score"),
        "title_score": None if score_info is None else score_info.get("title_score"),
        "artist_score": None if score_info is None else score_info.get("artist_score"),
        "album_score": None if score_info is None else score_info.get("album_score"),
        "duration_score": None if score_info is None else score_info.get("duration_score"),
        "duration_diff_sec": None if score_info is None else score_info.get("duration_diff_sec"),
        "error": error_msg,
    }

    if matched is None:
        base.update({
            "lrclib_id": None,
            "matched_track": None,
            "matched_artist": None,
            "matched_album": None,
            "matched_duration": None,
            "instrumental": None,
            "plain_lyrics": None,
            "synced_lyrics": None,
            "has_plain": False,
            "has_synced": False,
        })
        return base

    lrclib_id = get_field(matched, "id")
    cand_title = get_field(matched, "trackName", "track_name", "name", "title")
    cand_artist = get_field(matched, "artistName", "artist_name", "artist")
    cand_album = get_field(matched, "albumName", "album_name", "album")
    cand_duration = get_field(matched, "duration")
    instrumental = get_field(matched, "instrumental")
    plain_lyrics = get_field(matched, "plainLyrics", "plain_lyrics")
    synced_lyrics = get_field(matched, "syncedLyrics", "synced_lyrics")

    base.update({
        "lrclib_id": lrclib_id,
        "matched_track": cand_title,
        "matched_artist": cand_artist,
        "matched_album": cand_album,
        "matched_duration": cand_duration,
        "instrumental": instrumental,
        "plain_lyrics": plain_lyrics,
        "synced_lyrics": synced_lyrics,
        "has_plain": bool(plain_lyrics),
        "has_synced": bool(synced_lyrics),
    })
    return base

In [ ]:
def resolve_one_song(row):
    exact = lrclib_get(
        title=row["Title"],
        artist=row["Artist"],
    )

    if exact:
        exact_score = score_candidate(row, exact)
        if exact_score["final_score"] >= MIN_ACCEPT_SCORE:
            return build_result_row(
                row=row,
                matched=exact,
                method="get_exact",
                score_info=exact_score,
                status="matched"
            )

    candidates = lrclib_search(
        title=row["Title"],
        artist=row["Artist"],
    )

    if not candidates:
        return build_result_row(
            row=row,
            matched=None,
            method="search_fallback",
            score_info=None,
            status="not_found"
        )

    scored = []
    for cand in candidates:
        s = score_candidate(row, cand)
        scored.append((cand, s))

    scored.sort(key=lambda x: x[1]["final_score"], reverse=True)
    best_cand, best_score = scored[0]

    status = "matched" if best_score["final_score"] >= MIN_ACCEPT_SCORE else "low_confidence"

    return build_result_row(
        row=row,
        matched=best_cand,
        method="search_fallback",
        score_info=best_score,
        status=status
    )

In [ ]:
def load_scp_paths(scp_path: str):
    paths = []
    with open(scp_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            path = parts[-1]
            paths.append(path)

    return paths


def parse_hooktheory_filename(audio_path: str):
    """
    e.g., 070-shake_guilty-conscience_kygz-KaPmKB.mp3
    """
    fname = Path(audio_path).name
    stem = Path(audio_path).stem

    parts = stem.split("_")
    if len(parts) < 2:
        raise ValueError(f"file name exception: {fname}")

    artist_raw, title_raw = parts[0], parts[1]

    artist = artist_raw.replace("-", " ").strip()
    title = title_raw.replace("-", " ").strip()

    return {
        "File": fname,
        "AudioPath": audio_path,
        "Artist": artist,
        "Title": title,
    }


def build_work_df_from_hooktheory_scp(scp_path: str):
    audio_paths = load_scp_paths(scp_path)

    rows = []
    bad_rows = []

    for p in tqdm(audio_paths, desc="Parsing HookTheory scp"):
        try:
            item = parse_hooktheory_filename(p)
            rows.append(item)
        except Exception as e:
            bad_rows.append({
                "AudioPath": p,
                "error": str(e)
            })

    work_df = pd.DataFrame(rows)
    bad_df = pd.DataFrame(bad_rows)

    print(f"success {len(work_df)} 条")
    print(f"fail {len(bad_df)} 条")

    return work_df, bad_df

In [ ]:
SCP_PATH = "/home/hbli/songformer/repo/SongFormer/runs/hx_train_hooktheory_v3/results/hooktheory_all_unique.scp"

work_df, bad_df = build_work_df_from_hooktheory_scp(SCP_PATH)

display(work_df.head())
display(bad_df.head())

In [ ]:
OUTPUT_CSV = "/home/z/下载/hooktheory_lyrics/hooktheory_lrclib_results.csv"
OUTPUT_JSONL = "/home/z/下载/hooktheory_lyrics/hooktheory_lrclib_results.jsonl"
CHECKPOINT_EVERY = 50

if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    done_files = set(existing["File"].astype(str).tolist())
    print(f"already have {len(existing)}, skip")
else:
    existing = pd.DataFrame()
    done_files = set()

todo_df = work_df[~work_df["File"].astype(str).isin(done_files)].reset_index(drop=True)
print("wait for processing: ", len(todo_df))

results = []

for idx, row in tqdm(todo_df.iterrows(), total=len(todo_df), desc="Fetching lyrics"):
    try:
        out = resolve_one_song(row)
    except Exception as e:
        out = build_result_row(
            row=row,
            matched=None,
            method="internal_error",
            score_info=None,
            status="error",
            error_msg=str(e)
        )

    results.append(out)

    # save
    if (idx + 1) % CHECKPOINT_EVERY == 0 or (idx + 1) == len(todo_df):
        new_df = pd.DataFrame(results)

        if len(existing) > 0:
            save_df = pd.concat([existing, new_df], ignore_index=True)
        else:
            save_df = new_df.copy()

        save_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

        with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
            for _, r in save_df.iterrows():
                f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

        print(f"[checkpoint] have save to:\n  {OUTPUT_CSV}\n  {OUTPUT_JSONL}")

In [ ]:
final_df = pd.read_csv(OUTPUT_CSV)
len(final_df[final_df['plain_lyrics'].notna()]), len(final_df)

### lyrics result check

In [ ]:
df = pd.read_csv("/home/z/下载/hooktheory_lyrics/hooktheory_lrclib_results.csv")

In [ ]:
df.columns

In [ ]:
df[(df['match_score'] < 0.6) & (df['instrumental'] == False)][['File', 'AudioPath', 'Title', 'Artist', 'Release', 'Duration', 'matched_track', 'matched_artist', 'matched_album',
       'matched_duration', 'status', 'match_score', 'instrumental']]

### lyrics embedding rename

In [6]:
import glob
from pathlib import Path

In [ ]:
npy_lst = glob.glob("/mnt/ssd/hbli/songformer/runs/SSL/ssl_lyrics/multilingual-e5-large/harmonixset/*.npy")
npy_name_lst = [i.split('/')[-1] for i in npy_lst]
npy_id_lst = [i.split('_')[1] for i in npy_name_lst]

In [ ]:
sfbench_lst = glob.glob("/mnt/ssd/hbli/datasets/songformer/songformbench/data/audios/HarmonixSet/*.wav")
sfbench_name_lst = [i.split('/')[-1] for i in sfbench_lst]
sfbench_id_lst = [i.split('_')[1] for i in sfbench_name_lst]

In [ ]:
npy_dir = Path("/mnt/ssd/hbli/songformer/runs/SSL/ssl_lyrics/multilingual-e5-large/harmonixset")
wav_dir = Path("/mnt/ssd/hbli/datasets/songformer/songformbench/data/audios/HarmonixSet")

rename_count = 0
skip_count = 0

for npy_path in npy_dir.glob("*.npy"):
    parts = npy_path.stem.split("_")

    prefix = parts[0]
    song_id = parts[1]

    if prefix == "HX" and song_id in set(sfbench_id_lst):
        new_name = "BHX_" + "_".join(parts[1:]) + ".npy"
        new_path = npy_path.with_name(new_name)

        if new_path.exists():
            print(f"[SKIP] already exist: {new_path.name}")
            skip_count += 1
            continue

        npy_path.rename(new_path)
        print(f"[RENAME] {npy_path.name} -> {new_path.name}")
        rename_count += 1

print(f"\nFinish: Rename {rename_count} Skip {skip_count} Files.")

In [7]:
npy_lst = glob.glob("/mnt/ssd/hbli/songformer/runs/SSL/ssl_lyrics/multilingual-e5-large/harmonixset/v2/*.npz")
npy_name_lst = [i.split('/')[-1] for i in npy_lst]
npy_id_lst = [i.split('_')[1] for i in npy_name_lst]

In [8]:
sfbench_lst = glob.glob("/mnt/ssd/hbli/datasets/songformer/songformbench/data/audios/HarmonixSet/*.wav")
sfbench_name_lst = [i.split('/')[-1] for i in sfbench_lst]
sfbench_id_lst = [i.split('_')[1] for i in sfbench_name_lst]

In [9]:
npy_dir = Path("/mnt/ssd/hbli/songformer/runs/SSL/ssl_lyrics/multilingual-e5-large/harmonixset/v2")
wav_dir = Path("/mnt/ssd/hbli/datasets/songformer/songformbench/data/audios/HarmonixSet")

rename_count = 0
skip_count = 0

for npy_path in npy_dir.glob("*.npz"):
    parts = npy_path.stem.split("_")

    prefix = parts[0]
    song_id = parts[1]

    if prefix == "HX" and song_id in set(sfbench_id_lst):
        new_name = "BHX_" + "_".join(parts[1:]) + ".npz"
        new_path = npy_path.with_name(new_name)

        if new_path.exists():
            print(f"[SKIP] already exist: {new_path.name}")
            skip_count += 1
            continue

        npy_path.rename(new_path)
        print(f"[RENAME] {npy_path.name} -> {new_path.name}")
        rename_count += 1

print(f"\nFinish: Rename {rename_count} Skip {skip_count} Files.")

[RENAME] HX_0453_mygirl.npz -> BHX_0453_mygirl.npz
[RENAME] HX_0777_lebump.npz -> BHX_0777_lebump.npz
[RENAME] HX_0707_halfwaygone.npz -> BHX_0707_halfwaygone.npz
[RENAME] HX_0047_chinacatsunflower.npz -> BHX_0047_chinacatsunflower.npz
[RENAME] HX_0436_loveactually.npz -> BHX_0436_loveactually.npz
[RENAME] HX_0873_readyornot.npz -> BHX_0873_readyornot.npz
[RENAME] HX_0329_youreajerk.npz -> BHX_0329_youreajerk.npz
[RENAME] HX_0783_lighton.npz -> BHX_0783_lighton.npz
[RENAME] HX_0279_thebreaks.npz -> BHX_0279_thebreaks.npz
[RENAME] HX_0798_lovelikethis.npz -> BHX_0798_lovelikethis.npz
[RENAME] HX_0231_rightthurr.npz -> BHX_0231_rightthurr.npz
[RENAME] HX_0153_lalaland.npz -> BHX_0153_lalaland.npz
[RENAME] HX_0098_floods.npz -> BHX_0098_floods.npz
[RENAME] HX_0965_wakingupinvegascalvin.npz -> BHX_0965_wakingupinvegascalvin.npz
[RENAME] HX_0558_allnightlong.npz -> BHX_0558_allnightlong.npz
[RENAME] HX_0743_iminloveiwannadoit.npz -> BHX_0743_iminloveiwannadoit.npz
[RENAME] HX_0272_teachmeho

In [ ]:
npy_id_lst

### drop .mp3 for hooktheory npy

In [ ]:
folder = Path("/mnt/ssd/hbli/songformer/runs/SSL/ssl_lyrics/multilingual-e5-large/hooktheory/")

for old_path in folder.glob("*.mp3.npy"):
    new_name = old_path.name.replace(".mp3.npy", ".npy")
    new_path = old_path.with_name(new_name)
    old_path.rename(new_path)

In [3]:
folder = Path("/mnt/ssd/hbli/songformer/runs/SSL/ssl_lyrics/multilingual-e5-large/hooktheory/v2/")

for old_path in folder.glob("*.mp3.npz"):
    new_name = old_path.name.replace(".mp3.npz", ".npz")
    new_path = old_path.with_name(new_name)
    old_path.rename(new_path)

### npz check

In [10]:
import numpy as np

In [12]:
data = np.load("/mnt/ssd/hbli/songformer/runs/SSL/ssl_lyrics/multilingual-e5-large/hooktheory/v2/zoe_deja-te-conecto_Abm_MLkAxak.npz", allow_pickle=False)
print(data.files)

['line_embs', 'stanza_embs', 'line_to_stanza', 'global_emb', 'num_lines', 'num_stanzas']


In [ ]:
data['line_embs'].shape

(95, 1024)